<a href="https://colab.research.google.com/github/akjallow/Electrical-Load-Demand-Prediction/blob/master/load_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, BatchNormalization, Dropout, GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger
from tensorflow.keras.optimizers import Adam

# Hyperparameters
SEQ_LEN = 96      # past hours (4 days lookback)
HORIZON = 24      # forecast horizon (next 24 hours)
BATCH_SIZE = 128
EPOCHS = 5
LR = 1e-3

# Paths to save models & scalers
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "hybrid_cnn_gru.h5")
SCALER_PATH = os.path.join(MODEL_DIR, "scalers.pkl")
CSVLOG = os.path.join(MODEL_DIR, "training_log.csv")
CHKPT = os.path.join(MODEL_DIR, "best_model.h5")


In [ ]:

# 0) Load Dataset

import pandas as pd

# Define column names (adjust to your dataset)
col_names = [
    "timestamp", "day_of_week", "hour_of_day", "is_weekend",
    "temperature", "is_holiday", "solar_generation", "electricity_demand"
]

# Load your CSV file (update path as needed)
df = pd.read_csv(
    "hourly data(2000-2023).csv",   # <--- replace with your file path
    names=col_names,
    header=0,
    parse_dates=["timestamp"]
)

# Sort by timestamp and set as index
df = df.sort_values("timestamp").set_index("timestamp")

print("Dataset loaded:", df.shape)
print(df.head())


In [ ]:



feature_cols = ["electricity_demand", "temperature", "day_of_week",
                "hour_of_day", "is_weekend", "is_holiday"]

df_model = df[feature_cols].copy()
# Handle missing values using forward fill, then drop any remaining NaNs
df_model = df_model.ffill().dropna()
  # handle missing values

# Sliding window function to create sequences
def create_multivariate_sequences(values, seq_length=SEQ_LEN, horizon=HORIZON):
    X, y = [], []
    for i in range(len(values) - seq_length - horizon + 1):
        X.append(values[i:i+seq_length, :])                    # input block
        y.append(values[i+seq_length:i+seq_length+horizon, 0]) # demand only
    return np.array(X), np.array(y)

values = df_model.values
X_all, y_all = create_multivariate_sequences(values)


In [ ]:

# 3) Train/Test Split & Scaling


# Split by time to avoid leakage
split_index = int(len(X_all) * 0.85)
X_train, X_test = X_all[:split_index], X_all[split_index:]
y_train, y_test = y_all[:split_index], y_all[split_index:]

# Scale input features
scaler = MinMaxScaler()
n_features = X_train.shape[2]
X_train_flat = X_train.reshape(-1, n_features)
scaler.fit(X_train_flat)

def apply_scaler(X, scaler):
    n_samples, seq_len, n_feats = X.shape
    return scaler.transform(X.reshape(-1, n_feats)).reshape(n_samples, seq_len, n_feats)

X_train_s = apply_scaler(X_train, scaler)
X_test_s  = apply_scaler(X_test, scaler)

# Scale demand target separately
demand_scaler = MinMaxScaler()
demand_scaler.fit(y_train.reshape(-1, 1))

y_train_s = demand_scaler.transform(y_train.reshape(-1,1)).reshape(y_train.shape)
y_test_s  = demand_scaler.transform(y_test.reshape(-1,1)).reshape(y_test.shape)


In [ ]:

# 4) Build Hybrid CNN–GRU Model

n_features = X_train.shape[2]

def build_hybrid(seq_len=SEQ_LEN, n_features=n_features, horizon=HORIZON, lr=LR):
    inp = Input(shape=(seq_len, n_features))

    # CNN layers: short-term patterns
    x = Conv1D(64, 3, padding="causal", activation="relu")(inp)
    x = BatchNormalization()(x)
    x = Dropout(0.15)(x)

    x = Conv1D(32, 3, padding="causal", activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.1)(x)

    # GRU layers: long-term dependencies
    x = GRU(64, return_sequences=True)(x)
    x = Dropout(0.15)(x)
    x = GRU(32)(x)
    x = Dropout(0.1)(x)

    # Dense output for horizon forecast
    out = Dense(horizon)(x)

    model = Model(inp, out)
    model.compile(optimizer=Adam(learning_rate=lr), loss="mse", metrics=["mae"])
    return model

model = build_hybrid()
model.summary()


# 5) Training with Callbacks


callbacks = [
    EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint(CHKPT, monitor="val_loss", save_best_only=True),
    CSVLogger(CSVLOG)
]

history = model.fit(
    X_train_s, y_train_s,
    validation_data=(X_test_s, y_test_s),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=2
)


In [ ]:

# 6) Evaluation (MAE + RMSE)


# Predictions
y_pred_s = model.predict(X_test_s, batch_size=BATCH_SIZE)

# Inverse scaling
y_test_inv = demand_scaler.inverse_transform(y_test_s.reshape(-1,1)).reshape(y_test.shape)
y_pred_inv = demand_scaler.inverse_transform(y_pred_s.reshape(-1,1)).reshape(y_pred_s.shape)

# Metrics
mae = mean_absolute_error(y_test_inv.flatten(), y_pred_inv.flatten())
rmse = np.sqrt(mean_squared_error(y_test_inv.flatten(), y_pred_inv.flatten()))  # fixed version

print(f" Test MAE: {mae:.3f}, RMSE: {rmse:.3f}")

In [ ]:

# 7) Plots

plt.rcParams.update({"font.size": 12, "figure.dpi": 120, "font.family": "serif"})

# a) Training History
plt.figure(figsize=(8,6))
plt.plot(history.history["loss"], label="Training Loss", linewidth=2)
plt.plot(history.history["val_loss"], label="Validation Loss", linewidth=2, linestyle="--")
plt.title("Training vs Validation Loss", fontsize=16, weight="bold")
plt.xlabel("Epochs")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# b) Forecast Example
plt.figure(figsize=(10,6))
plt.plot(y_test_inv[0], label="Actual Demand", marker="o", linewidth=2)
plt.plot(y_pred_inv[0], label="Forecast Demand", marker="x", linewidth=2)
plt.title("24h Forecast Example (Test Sample 0)", fontsize=16, weight="bold")
plt.xlabel("Hour")
plt.ylabel("Electricity Demand (MW)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# c) Scatter Plot
plt.figure(figsize=(7,7))
plt.scatter(y_test_inv.flatten(), y_pred_inv.flatten(), alpha=0.3, s=10, color="navy")
plt.plot([y_test_inv.min(), y_test_inv.max()],
         [y_test_inv.min(), y_test_inv.max()], 'r--', linewidth=2)
plt.title("Actual vs Predicted Demand", fontsize=16, weight="bold")
plt.xlabel("Actual (MW)")
plt.ylabel("Predicted (MW)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# d) Residuals
residuals = y_test_inv.flatten() - y_pred_inv.flatten()
plt.figure(figsize=(10,5))
plt.hist(residuals, bins=50, color="purple", alpha=0.7, edgecolor="black")
plt.title("Residual Distribution", fontsize=16, weight="bold")
plt.xlabel("Error (MW)")
plt.ylabel("Frequency")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:

# 8) Save Model & Scalers

model.save("models/hybrid_cnn_gru.keras")

with open(SCALER_PATH, "wb") as f:
    pickle.dump({
        "feature_scaler": scaler,
        "demand_scaler": demand_scaler,
        "feature_cols": feature_cols,
        "seq_len": SEQ_LEN,
        "horizon": HORIZON
    }, f)

print(" Model and scalers saved:", MODEL_PATH, SCALER_PATH)

In [ ]:

!pip install streamlit pyngrok --quiet

import threading
import subprocess
from pyngrok import ngrok


NGROK_AUTH_TOKEN = "3A6l6EvtCQto2jclgohCqhOALkm_2SJEMgaXTf7LWEKVSKSEP"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)


def run_streamlit():
    # Force streamlit to run on port 8501
    subprocess.run(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])

# Run Streamlit in a separate thread
thread = threading.Thread(target=run_streamlit)
thread.start()



public_url = ngrok.connect(8501)
print("Streamlit app is live at:", public_url)
